# Credit Card Fraud Detection — EDA and Modeling
This notebook uses the same modular source pipeline as the Streamlit app. It never applies SMOTE to the test set.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocess import load_data, prepare_splits
from src.train_iforest import train_isolation_forest, iforest_scores
from src.train_xgb import train_xgboost
from src.evaluate import choose_threshold, evaluate_scores


In [ ]:
df, source = load_data('../data/creditcard.csv')
print(source, df.shape)
display(df.head())
print('Nulls:', int(df.isna().sum().sum()), 'Duplicates:', int(df.duplicated().sum()))
display(df.Class.value_counts().rename(index={0:'Legitimate',1:'Fraud'}))


In [ ]:
sns.countplot(data=df, x='Class', hue='Class', palette={0:'#00D9B5',1:'#FF4D6D'}, legend=False)
plt.title('Class imbalance'); plt.show()
sns.boxplot(data=df.sample(min(10000,len(df)), random_state=42), x='Class', y='Amount', hue='Class', showfliers=False)
plt.yscale('log'); plt.title('Amount by class (log scale)'); plt.show()


In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(15,12)); sns.heatmap(corr, cmap='vlag', center=0); plt.title('Feature correlation'); plt.show()
display(corr['Class'].drop('Class').abs().sort_values(ascending=False).head(12))


In [ ]:
X_train, X_test, y_train, y_test, scaler = prepare_splits(df)
print('Train:', y_train.value_counts().to_dict(), 'Test:', y_test.value_counts().to_dict())
# Validation partition comes only from the training partition.
from sklearn.model_selection import train_test_split
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=.2, stratify=y_train, random_state=42)
iso = train_isolation_forest(X_fit, contamination=float(df.Class.mean()))
iso_threshold = choose_threshold(y_val, iforest_scores(iso, X_val), min_precision=.10)
iso_metrics = evaluate_scores(y_test, iforest_scores(iso, X_test), iso_threshold)
xgb, X_smote, y_smote = train_xgboost(X_fit, y_fit)
xgb_threshold = choose_threshold(y_val, xgb.predict_proba(X_val)[:,1], min_precision=.10)
xgb_metrics = evaluate_scores(y_test, xgb.predict_proba(X_test)[:,1], xgb_threshold)
print('SMOTE training counts:', y_smote.value_counts().to_dict())
display(pd.DataFrame({'Isolation Forest':{k:iso_metrics[k] for k in ['recall','precision','f1','pr_auc','roc_auc','accuracy']}, 'XGBoost + SMOTE':{k:xgb_metrics[k] for k in ['recall','precision','f1','pr_auc','roc_auc','accuracy']}}))
